# this notebook hopes to achieve the following:
### 1) Create unique geo-linked cohort (for descriptives)
### 2) separete records with permission but did not link (for descriptives)


### output
### .json file for start/end date for each LSOA for each ID, from Cohort reported data.

In [ ]:
import pandas as pd
import os
import numpy as np

In [ ]:
os.chdir(r"S:\LLC_0002\lamj\notebooks_lsoa")

df_perm = pd.read_csv(r'S:\LLC_0002\lamj\notebooks_lsoa\LSOA\demographics_geo.csv')
df_hh = pd.read_stata("S:\LLC_0002\data\stata_w_labs\geo_air_pollution_hh_v0001_20231109.dta")
df_pc = pd.read_stata("S:\LLC_0002\data\stata_w_labs\geo_air_pollution_pc_v0001_20231109.dta")

In [ ]:
# check perm
df_perm['geocoding_permission'].value_counts()

"""
# checking if there are no records without permission
df_geo = df_perm[(df_perm['geocoding_permission'] != "0")]
df_geo = df_geo[(df_geo['geocoding_permission'] != "NULL")]
df_geo['geocoding_permission'].value_counts()
"""
df_perm

In [ ]:
# minimisation
columns_to_keep = ["llc_0002_stud_id","cohort","geocoding_permission"]
df_perm = df_perm[columns_to_keep]



In [ ]:
df_perm['llc_0002_stud_id']= df_perm['llc_0002_stud_id'].astype(np.int64)

In [ ]:
# data minimisation
# from each of hh and pc, keep only ID, LSOA, address_end_date, address_start_date

columns_to_keep = ['address_end_date','address_start_date','llc_0002_stud_id','lsoa11_e']
df_hh = df_hh[columns_to_keep]
df_pc = df_pc[columns_to_keep]


In [ ]:
### merge to check ovelapping
df_combined = pd.concat([df_hh,df_pc])
df_combined = df_combined.sort_values(by= 'llc_0002_stud_id')        
# inspect duplications.
df_combined

In [ ]:
# for each ID, count if unique (to exclude fully duplicated records by ID, LSOA, start & end date)

df_unique = df_combined.drop_duplicates(subset=['llc_0002_stud_id','lsoa11_e','address_start_date','address_end_date'], keep = False)
order = ['llc_0002_stud_id','lsoa11_e','address_start_date','address_end_date']
df_unique = df_unique[order]
df_unique

In [ ]:
df_unique = df_unique.sort_values(by = ['llc_0002_stud_id', 'address_start_date'])


In [ ]:
df_perm.dtypes

In [ ]:
# change type to string (to match with other datasets)
df_perm['llc_0002_stud_id'] = df_perm['llc_0002_stud_id'].apply(lambda x: str(x))

In [ ]:
# IF ID not in perm, remove.
df_geo = pd.merge(df_unique, df_perm, on = 'llc_0002_stud_id', how = 'left', suffixes = ('', '_perm'))
df_geo

In [ ]:
# sorting df
df_geo['address_start_date'] = pd.to_datetime(df_geo['address_start_date'], format = '%Y%m%d', errors ='coerce')
df_geo['address_end_date'] = pd.to_datetime(df_geo['address_end_date'], format = '%Y%m%d', errors ='coerce')
df_geo = df_geo.sort_values(by = ['llc_0002_stud_id', 'address_start_date'])
df_geo

In [ ]:
# Drop if address no start date
df_geo['address_start_date'].value_counts(dropna= False)



In [ ]:
# n records does not have start date... remove 
df_geo = df_geo.dropna(subset = ['address_start_date'])

In [ ]:
df_geo.sort_values(by = ["llc_0002_stud_id", 'address_start_date'], inplace = True)
df_geo['lsoa_number'] = df_geo.groupby('llc_0002_stud_id').cumcount() + 1
df_geo['lsoa_number'].value_counts()

In [ ]:
df_geo['lsoa_change_count'] = df_geo.groupby('llc_0002_stud_id')['lsoa_number'].transform('nunique') - 1 

In [ ]:
unique = df_geo.drop_duplicates(subset=  'llc_0002_stud_id')
unique['lsoa_change_count'].value_counts().sort_index()

In [ ]:
# keep records with only 1 LSOA for later descriptives
unique[unique['lsoa_change_count']==0].to_csv("one_LSOA_cohort.csv")

In [ ]:
unique[unique['lsoa_change_count']==0]['lsoa11_e'].value_counts()

In [ ]:
# assess range of start dates in this cohort
df_geo['address_start_date'].describe()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import datetime

plt.figure(figsize= (10,6))
plt.hist(df_geo['address_start_date'], bins = 200, edgecolor = 'black')
#plt.xlim(10000, 20000)
plt.show()

In [ ]:
df_geo['total_address_count'] = df_geo.groupby('llc_0002_stud_id')['lsoa11_e'].transform('count')



In [ ]:
df_geo['lsoa_change'] = (df_geo['lsoa11_e'] != df_geo['lsoa11_e'].shift()) & (df_geo.groupby('llc_0002_stud_id').cumcount() > 0)
df_geo.loc[df_geo.groupby('llc_0002_stud_id').head(1).index, 'lsoa_change'] = True

In [ ]:
df_geo[df_geo['lsoa_change']== False].count()

In [ ]:
df_geo = df_geo[df_geo["lsoa_change"]]
df_geo

In [ ]:
df_geo['lsoa11_e'].describe()

In [ ]:
df_geo['llc_0002_stud_id'].describe()

In [ ]:
# re-format dates as dates, not datetime.
# such that it would work as a json file.
# added step of converting date to string, and string to date.

df_geo['address_start_date'] = df_geo['address_start_date'].dt.normalize()
df_geo['address_end_date'] = df_geo['address_end_date'].dt.normalize()



In [ ]:
df_geo['address_start_date'].dtype

In [ ]:
from datetime import datetime
import numpy as np

df_geo['address_start_date'] = np.datetime_as_string(df_geo['address_start_date'], unit = 'D')
df_geo['address_end_date'] = np.datetime_as_string(df_geo['address_end_date'], unit = 'D')



In [ ]:
df_geo['address_start_date'].min()

In [ ]:
# to csv

unique = df_geo.drop_duplicates(subset=  'llc_0002_stud_id')
columns_to_keep = ["llc_0002_stud_id", 'lsoa_change_count']
unique = unique[columns_to_keep]
unique.to_csv("clean_cohort_id.csv")

In [ ]:
unique

In [ ]:
from collections import defaultdict

result = defaultdict(lambda: {'lsoa11_e': [], 'start_date':[], 'end_date':[], 'total_addresses':[]})


    
    


In [ ]:
for _, row in df_geo.iterrows():
    llc_0002_stud_id = row['llc_0002_stud_id']
    result[llc_0002_stud_id]['total_addresses'].append(row['total_address_count'])
    result[llc_0002_stud_id]['lsoa11_e'].append(row['lsoa11_e'])
    result[llc_0002_stud_id]['start_date'].append(row['address_start_date'])
    result[llc_0002_stud_id]['end_date'].append(row['address_end_date'])
    

In [ ]:
result = dict(result)

In [ ]:
result

In [ ]:
df_geo['address_start_date'].max()

In [ ]:
df_geo['address_start_date'].value_counts()

In [ ]:
df_geo['address_end_date'].describe()

In [ ]:
df_geo['address_end_date'].value_counts()

In [ ]:
# save dictionary - json
import json

with open("cohort_geo.json", "w") as outfile:
    json.dump(result, outfile)